<h1 align="center" style="color:#5e614b;">Deep Learning - Project</h1>
<h3 align="center" style="color:#5e614b;">Group 4 - Painting Classification</h3>

---

### <span style="color:#5e614b;">Group Members</span>

<table>
  <thead>
    <tr>
      <th>Name</th>
      <th>Student ID</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>Beatriz Pinto</td>
      <td>20250364</td>
    </tr>
    <tr>
      <td>Carlota Pires</td>
      <td>20250383</td>
    </tr>
    <tr>
      <td>Francisca Calçoa</td>
      <td>20240266</td>
    </tr>
    <tr>
      <td>Francisca Martins</td>
      <td>20250347</td>
    </tr>
    <tr>
      <td>Marta Feital</td>
      <td>20250407</td>
    </tr>
  </tbody>
</table>

---

<h2 style="color:#5e614b;">Dataset Exploration & Preprocessing Notebook</h2>

<p>
TO DO: contexto mínimo deste notebook
</p>

---

<a id="toc"></a>

<h2 style="color:#5e614b;">Table of Contents</h2>

[1. Imports and Setup](#imports)  
[2. Dataset Structure & Image Analysis](#analysis)  
Aqui fazer o indice

---

## <span id="imports" style="color:#5e614b;">1. Imports and Setup</span>

<p><a href="#toc" >Shortcut to Table of Contents</a></p>

In [ ]:
import numpy as np
import pandas as pd
import os
import hashlib
from numpy import ndarray
from pathlib import Path
import random 
import shutil


# Image Processing
from PIL import Image

# Visualization
import matplotlib.pyplot as plt

# Deep Learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.models import Model, Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Input
from keras.layers import LeakyReLU
from keras.utils import to_categorical
from keras.optimizers import SGD
from keras.losses import CategoricalCrossentropy
from keras.metrics import CategoricalAccuracy, AUC, F1Score
from keras.callbacks import ModelCheckpoint, CSVLogger, LearningRateScheduler
from keras.ops import add
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# Dataset
from keras.datasets.cifar10 import load_data as load_cifar10

>Alternative working directory paths for different environments:

In [ ]:
# os.chdir("/Users/marta/Downloads/deep_learning_project")
# os.chdir("/Users/carlo/OneDrive/Documentos/Mestrado/2S/DL/project/deep_learning_project")
# print("Working directory:", os.getcwd())
#os.chdir("/Users/calco/OneDrive/Ambiente de Trabalho/nova ims/2 Semestre/Projeto_DL/deep_learning_project")


In [ ]:
# Set-up data 

# Dataset path
#dataset_path = "../data/wikiart"
dataset_path = "data/wikiart"


# Loop over each artist folder inside the dataset
for artist in os.listdir(dataset_path):
    artist_path = os.path.join(dataset_path, artist)

    # Check if path is actually a folder and not a file
    if os.path.isdir(artist_path):

        # Loop over each image file inside the artist folder
        for image_file in os.listdir(artist_path):
            image_path = os.path.join(artist_path, image_file)

            # Print the image path for inspection/debugging
            print(image_path)

---

## <span id="analysis" style="color:#5e614b;">2. Dataset Structure & Image Analysis</span>

<p><a href="#toc" >Shortcut to Table of Contents</a></p>

> "[...] the presence of duplicated images in the training set not only negatively affects the efficiency of model training but also may result in lower accuracy of the image classifier."  
> — Aghabagherloo et al., 2025 

<h4 style="color:#5e614b;">
Let's begin by checking if we have duplicated images.
</h4> 

We'll begin by identifying the duplicates. We'll go through all the folders, read every image, and generate a unique fingerprint (hash) based on its raw pixel data.   
> https://docs.python.org/3/library/hashlib.html

We'll compare these hashes to detect when two images are exactly the same, even if they have different filenames or are stored in different folders. Then, we'll store the detected duplicate pairs for further inspection.
After finding these duplicates, we'll visualize them side-by-side to manually verify the results. At the same time, we'll keep track of which images will potentially be removed from the dataset.  
  
<span style="color:#5e614b; font-weight:bold;">Note:</span>
This code cell will stay commented because it's just for visual analysis, next is the code that actually removes the duplicates.

In [ ]:
# https://docs.python.org/3/library/hashlib.html

"""
seen_hashes = {} # stores {hash: image_path} for all images seen so far
duplicates = []
remove_images=[]

for artist in os.listdir(dataset_path):
    artist_path = os.path.join(dataset_path, artist)
    if os.path.isdir(artist_path):
        for img_file in os.listdir(artist_path):
            img_path = os.path.join(artist_path, img_file)
            #open image in binary mode and compute its unique fingerprint (hash)
            with open(img_path, "rb") as f:
                #see based on the pixels if the image is similiar
                img_hash = hashlib.sha256(f.read()).hexdigest()
            # if this hash was seen before, it will append in duplicates
            if img_hash in seen_hashes:
                duplicates.append({
                    "file": img_path,
                    "duplicate_of": seen_hashes[img_hash]
                })
            # if not store the hash in the dic to compare to the future images
            else:
                seen_hashes[img_hash] = img_path
            


fig, axes = plt.subplots(len(duplicates), 2, figsize=(10, 3 * len(duplicates)))

# if only one duplicate, make axes iterable
if len(duplicates) == 1:
    axes = [axes]

for i, d in enumerate(duplicates):
    for j, img_path in enumerate([d["file"], d["duplicate_of"]]):
        img = Image.open(img_path)
        
        axes[i][j].imshow(img)
        axes[i][j].set_title(img_path, fontsize=6)
        axes[i][j].axis("off")
    remove_images.append(d["duplicate_of"])

plt.suptitle("Duplicate Images", fontsize=14)
plt.tight_layout()
plt.show()

"""

There are two duplicated paintings by the artist **Child Hassam**. We will remove them. #FLAG qual deles é que removemos ou são os dois?

In [ ]:
"""
for path in remove_images:
    os.remove(path)
"""

We notice that these two duplicated paintings have the same name, but one of them ends with -1. So we will check if there are more paintings like these.

In [ ]:
"""
for artist in os.listdir(dataset_path):
    artist_path = os.path.join(dataset_path, artist)
    
    if os.path.isdir(artist_path):
        # find all images ending with -1.jpg
        duplicates = [f for f in os.listdir(artist_path)
                      if f.lower().endswith("-1.jpg")]
        
        for dup in duplicates:
            # build both file names
            dup_path = os.path.join(artist_path, dup)
            original_name = dup.replace("-1.jpg", ".jpg")
            original_path = os.path.join(artist_path, original_name)
            
            # only show if both exist
            if os.path.exists(original_path):
                fig, axes = plt.subplots(1, 2, figsize=(10, 5))
                
                axes[0].imshow(Image.open(original_path))
                axes[0].set_title(f"Original\n{original_name}")
                axes[0].axis("off")
                
                axes[1].imshow(Image.open(dup_path))
                axes[1].set_title(f"Duplicate (-1)\n{dup}")
                axes[1].axis("off")
                
                fig.suptitle(artist)
                plt.show()
"""            


In [ ]:
artist_duplicate_count = {}

for artist in os.listdir(dataset_path):
    artist_path = os.path.join(dataset_path, artist)
    
    if os.path.isdir(artist_path):
        count = 0
        duplicates = [f for f in os.listdir(artist_path)
                      if f.lower().endswith("-1.jpg")]
        
        for dup in duplicates:
            original_name = dup.replace("-1.jpg", ".jpg")
            original_path = os.path.join(artist_path, original_name)
            
            # only count if both original AND duplicate exist
            if os.path.exists(original_path):
                count += 1
        
        if count > 0:
            artist_duplicate_count[artist] = count

# print results
for artist, count in sorted(artist_duplicate_count.items(), key=lambda x: x[1], reverse=True):
    print(f"{artist}: {count} duplicated paintings")


After analysing all the paintings, we found 5 pairs of duplicated paintings, where the difference is just the zoom or the color tone . The rest of them were clearly different artworks and were kept.

In [ ]:
#duplicated_paitings=['../data/wikiart/Camille_Pissarro/camille-pissarro_sunset-bazincourt-steeple',
#                    '../data/wikiart/Childe_Hassam/childe-hassam_in-the-garden-1889',
#                    '../data/wikiart/Edgar_Degas/edgar-degas_alexander-and-bucephalus-detail-1861',
#                    '../data/wikiart/Edgar_Degas/edgar-degas_dancers-1899',
#                    '../data/wikiart/Nicholas_Roerich/nicholas-roerich_kneeling-warriors-study-of-murals-for-the-chapel-in-pskov-1914',
#                    '../data/wikiart/Vincent_van_Gogh/vincent-van-gogh_sien-sewing-half-figure-1883'
#                    ]
duplicated_paitings = [
    'data/wikiart/Camille_Pissarro/camille-pissarro_sunset-bazincourt-steeple',
    'data/wikiart/Childe_Hassam/childe-hassam_in-the-garden-1889',
    #'data/wikiart/Edgar_Degas/edgar-degas_alexander-and-bucephalus-detail-1861',
    'data/wikiart/Edgar_Degas/edgar-degas_dancers-1899',
    'data/wikiart/Nicholas_Roerich/nicholas-roerich_kneeling-warriors-study-of-murals-for-the-chapel-in-pskov-1914',
    'data/wikiart/Vincent_van_Gogh/vincent-van-gogh_sien-sewing-half-figure-1883'
]

paint_path=[]
for paint in duplicated_paitings:
    paint_og= paint+'.jpg'
    paint_dup= paint + '-1.jpg'
    paint_path.append((paint_og,paint_dup))
    # paint_path.append(paints_dup)   

In [ ]:
"""
for p,d in paint_path:
    
    fig, axes = plt.subplots(1, 2, figsize=(20, 5))
    
    axes[0].imshow(Image.open(p))
    axes[0].set_title(f"Original\n{p}")
    axes[0].axis("off")
    
    axes[1].imshow(Image.open(d))
    axes[1].set_title(f"Duplicate (-1)\n{d}")
    axes[1].axis("off")
    
    
    plt.show()
"""

Knowing that in **Augumentation** will already create tone variations of this paintings there is no reason for keeping them. So we will remove them.

For the case of the painting of third and last painting we will keep the paintgs where their names end with -1 and the rest we will keep the original name.

In [ ]:
"""
keep_duplicate = [2, 4]  # edgar-degas_alexander and nicholas-roerich_kneeling

for i, (p, d) in enumerate(paint_path):
    
    if i in keep_duplicate:
        os.remove(p)   # remove original, keep -1
        
    else:
        os.remove(d)  # remove -1, keep original
"""


In [ ]:
# check which extensions of images are there in the dataset

data_directory = Path("dataset_path")

set(
    img.suffix.lower()
    for class_folder in Path(dataset_path).iterdir()
    if class_folder.is_dir()
    for img in class_folder.glob("*")
    if img.is_file()
    )

This means the dataset only has .jpg images. yey!

In [ ]:
artist_dic={}
for artist in os.listdir(dataset_path):
    artist_path = os.path.join(dataset_path, artist)
    
    if os.path.isdir(artist_path):
        
        images = [f for f in os.listdir(artist_path)
                  if f.lower().endswith((".jpg"))]
        artist_dic[artist]= len(images)

In [ ]:
fig, ax = plt.subplots()
artist_dic = dict(sorted(artist_dic.items(), key=lambda x: x[1], reverse=True))
artist = list(artist_dic.keys())
painting = list(artist_dic.values())


ax.bar(artist, painting, color= '#5e614b')

ax.set_ylabel('Number of paintings')
ax.set_title('Number of paintings by Artist')

plt.xticks(rotation=90)
plt.show()

In [ ]:
len(artist_dic)

There are 23 artists in our datasets, where the ones with the most paitings are **Vicent Van Gogh** and **Nicholar Roerich** with 1322 and 1274, respectively.

In [ ]:
# collect sizes per artist
artist_sizes = {}

for artist in os.listdir(dataset_path):
    artist_path = os.path.join(dataset_path, artist)
    
    if os.path.isdir(artist_path):
        widths = []
        heights = []
        for img_file in os.listdir(artist_path):
            img_path = os.path.join(artist_path, img_file)
            try:
                with Image.open(img_path) as img:
                    w, h = img.size
                    widths.append(w)
                    heights.append(h)
            except:
                pass  # skip corrupted images
        
        if widths:
            artist_sizes[artist] = {
                "avg_width": np.mean(widths),
                "avg_height": np.mean(heights)
            }

# plot
artists = list(artist_sizes.keys())
avg_widths = [artist_sizes[a]["avg_width"] for a in artists]
avg_heights = [artist_sizes[a]["avg_height"] for a in artists]

x = np.arange(len(artists))
fig, ax = plt.subplots(figsize=(15, 6))

ax.bar(x - 0.2, avg_widths, 0.4, label='Avg Width', color="#c1c0be")
ax.bar(x + 0.2, avg_heights, 0.4, label='Avg Height', color= '#5e614b')

ax.set_xticks(x)
ax.set_xticklabels(artists, rotation=90)
ax.set_ylabel('Pixels')
ax.set_title('Average Image Size per Artist')
ax.legend()

#tight_layout()
plt.show()

In [ ]:
# check which variations there are between images
# (size, aspect ratio, and color mode)
# This helps ensure consistency before feeding data into a CNN


# Loop through each class folder (e.g., Monet, Vangogh, etc.)
for artist in os.listdir(dataset_path):
    artist_path = os.path.join(dataset_path, artist)

    if os.path.isdir(artist_path):
        widths = []
        heights = []
        aspect_ratio=[]
        mod=[]
        for img_file in os.listdir(artist_path):
            img_path = os.path.join(artist_path, img_file)
            try:
                with Image.open(img_path) as img:
                    w, h = img.size # image dimensions
                    widths.append(w)
                    heights.append(h)
                    aspect_ratio.append(round(w / h, 2)) # shape of the image
                    mod.append(img.mode)                # color format (e.g., RGB, L)
            except:
                pass  # skip corrupted images
        
        if widths:
            # Store metadata in dictionary using file name as key 
            artist_sizes[artist] = {
                "avg_width": np.mean(widths),
                "avg_height": np.mean(heights),
                "avg_aspectRatio": np.mean(aspect_ratio),
                "avg_mode": pd.Series(mod).mode()[0]  
            }



# build the table
df_summary = pd.DataFrame([
    {
        "Artist": artist,
        "Avg Width": round(artist_sizes[artist]["avg_width"], 1),
        "Avg Height": round(artist_sizes[artist]["avg_height"], 1),
        "Avg Aspect Ratio": round(artist_sizes[artist]["avg_aspectRatio"], 3),
        "Most Common Mode": artist_sizes[artist]["avg_mode"]
    }
    for artist in artist_sizes
])

df_summary.set_index("Artist", inplace=True)
display(df_summary)

In [ ]:
from collections import Counter

mod_counts = Counter(mod)
mod_counts

## Outlier Detection

To check if there is some anomalie in the dataset, we will check for outliers with:

- **Isolation Forest with ResNet50**

**Isolation Forest** is an algorithm for anomaly detection, where they use trees that randomly split features and which gives two outputs:

- 1 -> normal painting
- -1 -> anomaly

- contamination: assumes 5% of data are anomalies

**ResNet50** is a CNN model used to extract feature vectors from images and works well with high dimensional. In terms of parameters we have:

- weights='imagenet', which the model uses pre trained weights from ImageNet
- include_top = False, removes the final classification
- pooling = 'avg', converts feature maps into a single vector


In [ ]:
paths  = []
labels = []

for artist in os.listdir(dataset_path):
    artist_path = os.path.join(dataset_path, artist)

    if os.path.isdir(artist_path):
        for f in os.listdir(artist_path):
            if f.lower().endswith(".jpg"):
                paths.append(os.path.join(artist_path, f))
                labels.append(artist)

print(f"Total images found: {len(paths)}")

base_model = ResNet50(
    weights='imagenet',
    include_top=False,      
    pooling='avg',          
    input_shape=(512, 512, 3)
)
base_model.trainable = False  

def load_image(path, size=(512, 512)):
    img = Image.open(path).convert('RGB')
    img = img.resize(size)
    return np.array(img) / 255.0  

embeddings  = []
paths_ok    = []   
labels_ok   = []

for path, label in zip(paths, labels):
    try:
        img      = load_image(path)                          
        img_proc = preprocess_input(img[np.newaxis] * 255.0) 
        emb      = base_model.predict(img_proc, verbose=0)  
        embeddings.append(emb[0])
        paths_ok.append(path)
        labels_ok.append(label)
    except Exception as e:
        print(f"Error loading {path}: {e}")  

embeddings = np.array(embeddings)
print(f"Embeddings extracted: {embeddings.shape}") 


scaler            = StandardScaler()
embeddings_scaled = scaler.fit_transform(embeddings)

iso = IsolationForest(
    n_estimators=100,
    contamination=0.05,  
    random_state=42
)
iso.fit(embeddings_scaled)

predictions = iso.predict(embeddings_scaled)          
scores      = iso.decision_function(embeddings_scaled) 

print(f"Outliers detected: {(predictions == -1).sum()} / {len(predictions)}")

sorted_idx  = np.argsort(scores)  
worst_20    = sorted_idx[:20]

plt.figure(figsize=(15, 8))
for i, idx in enumerate(worst_20):
    img = Image.open(paths_ok[idx]).convert('RGB').resize((256, 256))
    plt.subplot(4, 5, i + 1)
    plt.imshow(img)
    plt.title(f"{labels_ok[idx]}\n{scores[idx]:.3f}", fontsize=7)
    plt.axis('off')

plt.suptitle('Inspect these images before removing', fontsize=12)
plt.tight_layout()
plt.show()


We will create a new folder where we can check all of the 667 outliers found.

In [ ]:
outlier_folder = "potential_outliers"
os.makedirs(outlier_folder, exist_ok=True)

for idx in np.where(predictions == -1)[0]:
    src_path = paths_ok[idx]
    
    artist = os.path.basename(os.path.dirname(src_path))
    filename = f"{artist}__{os.path.basename(src_path)}"
    
    dst_path = os.path.join(outlier_folder, filename)
    shutil.copy(src_path, dst_path)

print(f"Copied {len(np.where(predictions == -1)[0])} outliers to '{outlier_folder}'")


After analysing 667 images, there is no any evidence to delete pictures. When we start the models we can check if there is improvement or not.

## Data Splitting

For the data splitting, we will use stratified split to guarantee that we have the same proportion of class in each dataset. We will use 70% for train, 15% for val and 15% for test set.

To create new folds we create a script called download_dataset_split.py to create news datasets. To run the script in the terminal we must write python scripts\download_dataset_split.py and it will create a new folder.

This part of the code will stay commented since we keep the files in Google Drive. 


In [ ]:
"""
dataset_path = Path("data/wikiart")
dest_root = Path("data_split")

train_pct = 0.7
val_pct = 0.15

splits = ["train", "val", "test"]

for split in splits:
    (dest_root / split).mkdir(parents=True, exist_ok=True)

def is_split_empty(dest_root, splits):
    return all(not list((dest_root / s).iterdir()) for s in splits)

if is_split_empty(dest_root, splits):
    print("Creating splits...")
    random.seed(123)

    for class_dir in sorted(dataset_path.iterdir()):
        if not class_dir.is_dir():
            continue

        images = [f for f in class_dir.iterdir() if f.is_file()]
        random.shuffle(images)

        n = len(images)
        n_train = int(n * train_pct)
        n_val   = int(n * val_pct)

        split_images = {
            "train": images[:n_train],
            "val":   images[n_train:n_train + n_val],
            "test":  images[n_train + n_val:]
        }

        for split_name, imgs in split_images.items():
            target_dir = dest_root / split_name / class_dir.name
            target_dir.mkdir(parents=True, exist_ok=True)
            for img in imgs:
                shutil.copy2(img, target_dir / img.name)

    print("Splits created successfully!")
else:
    print("Splits already exist, skipping.")

print("\n── Diagnostic ──")
for split in splits:
    class_dirs = [d for d in (dest_root / split).iterdir() if d.is_dir()]
    print(f"{split}: {len(class_dirs)} class folders")
    if class_dirs:
        example_files = list(class_dirs[0].iterdir())[:3]
        print(f"  Example files: {[f.name for f in example_files]}")

VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def load_data(folder, encoder=None):
    folder = Path(folder)
    paths, labels = [], []

    for class_dir in sorted(folder.iterdir()):
        if not class_dir.is_dir():
            continue
        for img_file in class_dir.iterdir():
            if img_file.is_file() and img_file.suffix.lower() in VALID_EXTENSIONS:
                paths.append(str(img_file))
                labels.append(class_dir.name)

    if encoder is None:
        encoder = LabelEncoder()
        labels = encoder.fit_transform(labels)
    else:
        labels = encoder.transform(labels)

    return paths, labels, encoder

train_paths, train_labels, encoder = load_data(dest_root / "train")
val_paths,   val_labels,   _       = load_data(dest_root / "val",  encoder=encoder)
test_paths,  test_labels,  _       = load_data(dest_root / "test", encoder=encoder)

print(f"\nLoaded {len(train_paths)} training samples.")
print(f"Loaded {len(val_paths)} validation samples.")
print(f"Loaded {len(test_paths)} testing samples.")
print(f"Number of classes: {len(encoder.classes_)}")
"""

In [ ]:
def get_image_paths_and_labels(base_path):
    base = Path(base_path)
    image_paths = list(base.rglob("*.jpg"))  
    data = {
        "filepath": [str(p) for p in image_paths],
        "label": [p.parent.name for p in image_paths]
    }
    return pd.DataFrame(data)


# Get the paths and labels for train, val, and test sets
dest_root = Path('.') / "data_split"
train_df = get_image_paths_and_labels(dest_root / 'train')
val_df = get_image_paths_and_labels(dest_root / 'val')
test_df = get_image_paths_and_labels(dest_root / 'test')

Change the code below

In [ ]:
# ── 1. Load ──────────────────────────────────────────────────────────────────
#sdataset_path = "../data/wikiart"
tf.keras.utils.set_random_seed(13)

train_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(128, 128),
    batch_size=32
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(128, 128),
    batch_size=32
)

# ── 2. Extract class info (before any transforms) ────────────────────────────
class_names = train_ds.class_names
num_classes  = len(class_names)

# ── 3. Define transforms ──────────────────────────────────────────────────────
normalization_layer = tf.keras.layers.Rescaling(1./255)

augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomTranslation(0.1, 0.1),
    tf.keras.layers.RandomContrast(0.2),
], name="augmentation")

# ── 4. Apply transforms ───────────────────────────────────────────────────────
AUTOTUNE = tf.data.AUTOTUNE

train_ds = (train_ds
    .map(lambda x, y: (normalization_layer(x), y),        # normalize once
         num_parallel_calls=AUTOTUNE)
    .map(lambda x, y: (augmentation(x, training=True), y), # augment only train
         num_parallel_calls=AUTOTUNE)
    .prefetch(AUTOTUNE))

val_ds = (val_ds
    .map(lambda x, y: (normalization_layer(x), y),        # normalize once
         num_parallel_calls=AUTOTUNE)
    .prefetch(AUTOTUNE))                                   # no augmentation

In [ ]:
'''data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])'''

In [ ]:
'''model = tf.keras.Sequential([
    #data_augmentation,
    tf.keras.layers.Conv2D(32, (3,3), activation="relu", input_shape=(128,128,3)),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, (3,3), activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(num_classes, activation="softmax")
])'''

In [ ]:
'''model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)'''

In [ ]:
#model.fit(train_ds, validation_data=val_ds, epochs=20)

Bibliography refrences # FLAG aprimorar isto
**Reference**

Aghabagherloo, A., Abadi, A., Sarkar, S., Dasu, V. A., & Preneel, B. (2025).  
*Impact of Data Duplication on Deep Neural Network-Based Image Classifiers: Robust vs. Standard Models.*  
2025 IEEE Security and Privacy Workshops (SPW), pp. 177–183.  
https://doi.org/10.1109/SPW67851.2025.00023